## **Installing Essential Packages**

In [ ]:
!!pip install -q langchain-groq

In [ ]:
!pip install langchain

In [ ]:
!pip install -U langchain langchain-core langchain-community langchain-text-splitters


In [ ]:
!pip install sentence-transformers  # required for HuggingFace embeddings
!pip install faiss-cpu

In [ ]:
!pip install pypdf

# **PDF Loader**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/LoveStories.pdf")
docs = loader.load()
print(len(docs), docs[0].metadata)

# **Tokenizer**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

print("chunks:", len(chunks))
# each element in `chunks` is a Document with .page_content and .metadata
print("first chunk content preview:", chunks[0].page_content[:300])


# **Embedding**

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Choose any HuggingFace embedding model
emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# **VectorDB and Retrieval**

In [ ]:
# Create FAISS index from your chunks
index = FAISS.from_documents(chunks, emb)
# Build retriever
retriever = index.as_retriever(search_kwargs={"k": 5})

results_with_scores = index.similarity_search_with_score("A STRANGE TWIST OF FATE ", k=5)
for doc, score in results_with_scores:
    print(doc.metadata.get("source"), score)


# **Gemini API Integration**

In [ ]:
from langchain_groq import ChatGroq
import os
import json

from google.colab import userdata
groq_key = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=groq_key
)

prompt = """
You are a movie database assistant.

Convert the user request into a structured JSON search query.
The JSON must contain:
- keywords
- genre
- year
- description

User request: "movies about aliens in 1980"

Return ONLY valid JSON.
"""

response = llm.invoke(prompt)

# Pretty print
try:
    data = json.loads(response.content)
    print(json.dumps(data, indent=4))
except:
    print(response.content)

# **LCEL Implementation**

In [ ]:
# minimal LCEL pipeline using index.similarity_search_with_score directly
import itertools
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0,     groq_api_key=groq_key)

# Runnable that calls the FAISS index directly and returns a mapping {context, question}
search_runnable = RunnableLambda(
    lambda inp: {
        "context": "\n\n".join(
            f"[source:{(d.metadata.get('source') if getattr(d, 'metadata', None) else '')}] {d.page_content[:2000]}"
            for d, _score in index.similarity_search_with_score(inp["question"], k=5)
        ),
        "question": inp["question"],
    }
)

prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Use ONLY the CONTEXT to answer the QUESTION.
If the answer is not in the context, reply "I don't know".

CONTEXT:
{context}

QUESTION: {question}

Answer concisely."""
)

pipeline = search_runnable | prompt | llm

resp = pipeline.invoke({"question": "Title of the chapters"})

# safe extraction
out = getattr(resp, "content", None) or getattr(resp, "text", None) or str(resp)
print(out.strip())


# **RetrievalQA**

In [ ]:
from langchain_classic.chains import RetrievalQA
from langchain_groq import ChatGroq
# Gemini LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",    # or llama-3.3-70b-versatile
    temperature=0,
    groq_api_key=groq_key
)
# Create QA chains
qa_stuff = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever
)
# Query
print(qa_stuff.run('Title of the chapters'))

In [ ]:
from langchain_classic.chains import RetrievalQA
from langchain_groq import ChatGroq
# Gemini LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",    # or llama-3.3-70b-versatile
    temperature=0,
    groq_api_key=groq_key
)
# Create QA chains
qa_stuff = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='map_reduce',
    retriever=retriever
)
# Query
print(qa_stuff.run('Title of the chapters'))

# **ConversationChain and ConversationBufferMemory**

In [ ]:
from langchain_groq import ChatGroq
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0,groq_api_key=groq_key)


memory = ConversationBufferMemory(memory_key="history", return_messages=True)

chat = ConversationChain(llm=llm, memory=memory, verbose=False)

print("Me   : Hi, my name is Shailesh.")
print("Bot  : ",chat.predict(input="Hi, my name is Shailesh."))
print("Me   : Remember that I like sci-fi movies.")
print("Bot  : "+chat.predict(input="Remember that I like sci-fi movies."))
print("Me   : What is my name and what do I like?")
answer = chat.predict(input="What is my name and what do I like?")
print("Bot  : "+answer)


# **Tool Calling**

In [ ]:
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import ToolMessage, HumanMessage

@tool
def add(a: int, b: int) -> int:
    """Return a + b."""
    return a + b

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,groq_api_key=groq_key
)

llm_with_tools = llm.bind_tools([add])

# ---- Step 1: user message ----
user_msg = HumanMessage(content="What is 12 + 9?")
response = llm_with_tools.invoke([user_msg])

# ---- Step 2: execute tool ----
tool_call = response.tool_calls[0]
result = add.invoke(tool_call["args"])

# ---- Step 3: send tool result back ----
tool_message = ToolMessage(
    content=str(result),
    tool_call_id=tool_call["id"],
)

final_response = llm_with_tools.invoke([
    user_msg,     # ✅ REQUIRED
    response,     # assistant tool request
    tool_message  # tool output
])

print(final_response.content)

# **Online RAG**

In [ ]:
from langchain_classic.schema import BaseRetriever, Document
from langchain_classic.chains import RetrievalQA
from langchain_groq import ChatGroq

import requests
from bs4 import BeautifulSoup
import os

# -----------------------------
# Fetch webpage text (LIMIT SIZE)
# -----------------------------
def fetch_text(url: str) -> str:
    headers = {"User-Agent": "Mozilla/5.0"}

    r = requests.get(url, headers=headers, timeout=10)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    for s in soup(["script", "style"]):
        s.decompose()

    text = "\n".join(
        line.strip()
        for line in soup.get_text().splitlines()
        if line.strip()
    )

    # ⭐ IMPORTANT: limit context size for Groq
    return text[:20000]   # keep only first ~20k chars


# -----------------------------
# Create Document
# -----------------------------
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
page_text = fetch_text(url)

web_doc = Document(
    page_content=page_text,
    metadata={"source": url}
)


# -----------------------------
# Custom Retriever
# -----------------------------
class WebOnlyRetriever(BaseRetriever):
    doc: Document

    def _get_relevant_documents(self, query: str):
        # return smaller context only
        return [self.doc]

    async def _aget_relevant_documents(self, query: str):
        return self._get_relevant_documents(query)


combined = WebOnlyRetriever(doc=web_doc)


# -----------------------------
# Groq LLM (FASTER + SAFER)
# -----------------------------
llm = ChatGroq(
    model="llama-3.1-8b-instant",   # ⭐ use this for demo
    temperature=0,
    max_tokens=512,
    groq_api_key=groq_key
)


# -----------------------------
# Retrieval QA Chain
# -----------------------------
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",   # ⭐ CRITICAL FIX (NOT map_reduce)
    retriever=combined
)


# -----------------------------
# Ask Question
# -----------------------------
result = qa.invoke({"query": "What is artificial intelligence?"})

print(result["result"])

# **Graph Rag**

In [ ]:
import requests
from langchain_groq import ChatGroq

def sparql(query):
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        # REQUIRED by Wikidata endpoint
        "User-Agent": "AI-GraphRAG-Demo/1.0 (research-demo@example.com)"
    }
    r = requests.get(url, params={"query": query}, headers=headers)
    r.raise_for_status() # Raise an exception for HTTP errors
    return r.json()

entity = "Q42"   # Douglas Adams

# retrieve graph neighbors
q = f"""
SELECT ?propLabel ?valueLabel WHERE {{
  wd:{entity} ?prop ?value .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
LIMIT 30
"""

try:
    graph = sparql(q)
    print(graph)
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import requests
import os
from langchain_groq import ChatGroq


# -------------------------------------------------
# 2. SPARQL Query Function (FIXED 403 ERROR)
# -------------------------------------------------
def sparql(query: str):
    url = "https://query.wikidata.org/sparql"

    headers = {
        "Accept": "application/sparql-results+json",
        # REQUIRED by Wikidata endpoint
        "User-Agent": "AI-GraphRAG-Demo/1.0 (research-demo@example.com)"
    }

    response = requests.get(
        url,
        params={"query": query},
        headers=headers,
        timeout=20
    )

    response.raise_for_status()
    return response.json()


# -------------------------------------------------
# 3. Select Entity (Douglas Adams)
# -------------------------------------------------
entity = "Q42"

query = f"""
SELECT ?propLabel ?valueLabel WHERE {{
  wd:{entity} ?prop ?value .
  SERVICE wikibase:label {{
      bd:serviceParam wikibase:language "en".
  }}
}}
LIMIT 30
"""


# -------------------------------------------------
# 4. Retrieve Knowledge Graph Neighbors
# -------------------------------------------------
graph = sparql(query)

triples = []
for row in graph["results"]["bindings"]:
    prop = row["propLabel"]["value"]
    value = row["valueLabel"]["value"]
    triples.append(f"{prop}: {value}")

graph_text = "\n".join(triples)

print("\n--- Retrieved Graph Facts ---\n")
print(graph_text)


# -------------------------------------------------
# 5. Initialize Groq LLM
# -------------------------------------------------
llm = ChatGroq(
    model="llama-3.1-8b-instant",   # fast + ideal for QA
    temperature=0,groq_api_key=groq_key
)


# -------------------------------------------------
# 6. Ask Question Using Graph Context
# -------------------------------------------------
prompt = f"""
You are a knowledge graph reasoning assistant.

Using ONLY the facts below, explain who Douglas Adams is.

Facts:
{graph_text}

Provide a concise 5-sentence summary.
"""

response = llm.invoke(prompt)

print("\n--- LLM Answer ---\n")
print(response.content)